In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import statsmodels.api as sm

processed_dir = Path("../data/processed")

# Load the datasets
consumption_analysis = pd.read_parquet(processed_dir / "consumption_analysis.parquet")
recommendations_clean = pd.read_parquet(processed_dir / "recommendations_clean.parquet")
ratings_clean = pd.read_parquet(processed_dir / "ratings_clean.parquet")
movies_clean = pd.read_parquet(processed_dir / "movies_clean.parquet")
model_df = consumption_analysis.copy()
model_df = model_df.rename(columns={"consumed_after_recommendation": "consumed"})

## Q3: Among recommended movies, which factors are associated with subsequent consumption?

Because `consumed` is a binary outcome (yes/no), we want to use logistic regression. Logistic regression is designed for binary outcomes and gives us a probability of consumption, allowing us to examine whether `predictedRating` is associated with the likelihood of a user watching the recommended movie while controlling for other relevant variables.

Some factors we may account for as controls include:

- **Movie popularity** — Popular movies may be consumed regardless of the recommendation.
- **User activity** — More active users may be more likely to consume or rate movies in general.
- **Genre** — Some genres may naturally have higher consumption rates.
- **Time / observation opportunity** — Recommendations made near the end of a user's observed history have less opportunity for a later rating.
- **Recommendation frequency** — Repeated exposure to the same movie may affect consumption differently from a first recommendation.

In [2]:
model_df.head()

,user_id,movie_id,recommendation_tstamp,predictedRating,recommendation_id,belief_tstamp,user_predict_rating,rating_tstamp,rating,time_to_consumption,consumed,prior_rating_tstamp,prior_rating,prior_consumption
0,377084,924,2023-03-01 06:10:51,4.303385,2,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
1,377084,1201,2023-03-01 06:10:51,4.215998,5,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
2,377084,1204,2023-03-01 06:10:51,4.236355,7,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
3,377084,1258,2023-03-01 06:10:51,4.241269,4,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
4,377084,1732,2023-03-01 06:10:51,4.255416,3,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False


In [3]:
model_df = model_df[["user_id", "movie_id", "predictedRating", "recommendation_tstamp", "consumed"]]

In [4]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed
0,377084,924,4.303385,2023-03-01 06:10:51,False
1,377084,1201,4.215998,2023-03-01 06:10:51,False
2,377084,1204,4.236355,2023-03-01 06:10:51,False
3,377084,1258,4.241269,2023-03-01 06:10:51,False
4,377084,1732,4.255416,2023-03-01 06:10:51,False


## Feature Engineering

In [5]:
# Feature: movie popularity
movie_popularity = (
    ratings_clean[ratings_clean["rating"] >= 0]
    .groupby("movie_id").size().rename("movie_popularity")
)

# Log-transform the movie popularity to reduce skewness
movie_popularity = np.log1p(movie_popularity)

movie_popularity.head()

movie_id
1    7.521859
2    6.876265
3    4.615121
4    3.367296
5    4.948760
Name: movie_popularity, dtype: float64

In [6]:
# Feature: user activity
user_activity = (
    ratings_clean[ratings_clean["rating"] >= 0]
    .groupby("user_id").size().rename("user_activity")
)

# Log-transform the user activity to reduce skewness
user_activity = np.log1p(user_activity)

user_activity.head()

user_id
42170    5.746203
43715    7.385231
44282    7.763021
50108    4.882802
50602    6.538140
Name: user_activity, dtype: float64

In [7]:
# Merge movie popularity and user activity into the model_df
model_df = model_df.merge(
    movie_popularity, 
    on="movie_id", 
how="left")

model_df = model_df.merge(
    user_activity,
    on="user_id",
    how="left"
)

In [8]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed,movie_popularity,user_activity
0,377084,924,4.303385,2023-03-01 06:10:51,False,7.218910,6.827629
1,377084,1201,4.215998,2023-03-01 06:10:51,False,6.855409,6.827629
2,377084,1204,4.236355,2023-03-01 06:10:51,False,6.171701,6.827629
3,377084,1258,4.241269,2023-03-01 06:10:51,False,7.410347,6.827629
4,377084,1732,4.255416,2023-03-01 06:10:51,False,7.380256,6.827629


In [9]:
model_null_count = model_df.isnull().sum()
print(f"Null counts in model_df:\n {model_null_count}")

Null counts in model_df:
 user_id                     0
movie_id                    0
predictedRating             0
recommendation_tstamp       0
consumed                    0
movie_popularity         5097
user_activity             144
dtype: int64


In [10]:
# Fill missing values with 0 for movie popularity and user activity
model_df["movie_popularity"] = model_df["movie_popularity"].fillna(0)
model_df["user_activity"] = model_df["user_activity"].fillna(0)

In [11]:
model_null_count = model_df.isnull().sum()
print(f"Null counts in model_df:\n {model_null_count}")

Null counts in model_df:
 user_id                  0
movie_id                 0
predictedRating          0
recommendation_tstamp    0
consumed                 0
movie_popularity         0
user_activity            0
dtype: int64


In [12]:
# Feature: primary genre
movies_clean.head()

,movie_id,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [13]:
# Take the first element of the genre list for each movie as the primary genre
movies_clean["primary_genre"] = movies_clean["genres"].apply(lambda x: x[0] if len(x) > 0 else None)

In [14]:
movies_clean["primary_genre"].value_counts()

primary_genre
Drama          24623
Comedy         21795
Action         11454
Documentary    10712
Unknown         9162
Horror          6080
Crime           4843
Adventure       3891
Animation       3812
Children        2599
Thriller        1819
Romance          962
Sci-Fi           859
Western          839
Mystery          682
Fantasy          631
War              160
Musical          109
Film-Noir         38
IMAX               1
Name: count, dtype: int64

In [15]:
# Group low frequency genres into an "Other" category
MIN_MOVIES = 1000  # threshold

genre_counts = movies_clean["primary_genre"].value_counts()
keep_genres = genre_counts[genre_counts >= MIN_MOVIES].index

movies_clean["genre_bucket"] = movies_clean["primary_genre"].where(
    movies_clean["primary_genre"].isin(keep_genres), "Other"
)

In [16]:
movies_clean["genre_bucket"].value_counts()

genre_bucket
Drama          24623
Comedy         21795
Action         11454
Documentary    10712
Unknown         9162
Horror          6080
Crime           4843
Other           4281
Adventure       3891
Animation       3812
Children        2599
Thriller        1819
Name: count, dtype: int64

In [17]:
# Merge the features into the model_df
model_df = model_df.merge(
    movies_clean[["movie_id", "genre_bucket"]],
    on="movie_id",
    how="left"
)

In [18]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed,movie_popularity,user_activity,genre_bucket
0,377084,924,4.303385,2023-03-01 06:10:51,False,7.218910,6.827629,Adventure
1,377084,1201,4.215998,2023-03-01 06:10:51,False,6.855409,6.827629,Action
2,377084,1204,4.236355,2023-03-01 06:10:51,False,6.171701,6.827629,Adventure
3,377084,1258,4.241269,2023-03-01 06:10:51,False,7.410347,6.827629,Horror
4,377084,1732,4.255416,2023-03-01 06:10:51,False,7.380256,6.827629,Comedy


In [19]:
recommendations_clean.head()

,user_id,tstamp,movie_id,predictedRating
0,377084,2023-03-01 06:10:51,296,4.626386
1,377084,2023-03-01 06:10:51,55820,4.343642
2,377084,2023-03-01 06:10:51,924,4.303385
3,377084,2023-03-01 06:10:51,1732,4.255416
4,377084,2023-03-01 06:10:51,1258,4.241269


In [20]:
# Feature: available time window for consumption
# The difference between current recommendation timestamp and next timestamp for same user-movie pairs

# Get the next recommendation timestamp for each user-movie pair
rec_sorted = recommendations_clean.sort_values(["user_id", "movie_id", "tstamp"])
rec_sorted["next_recommendation_tstamp"] = (
    rec_sorted.groupby(["user_id", "movie_id"])["tstamp"].shift(-1)
)

rec_sorted = rec_sorted.rename(columns={"tstamp": "recommendation_tstamp"})

# Merge the next recommendation timestamp into the model_df
model_df = model_df.merge(
    rec_sorted[["user_id", "movie_id", "recommendation_tstamp", "next_recommendation_tstamp"]],
    on=["user_id", "movie_id", "recommendation_tstamp"],
    how="left"
)

dataset_end = ratings_clean["tstamp"].max()  # Fall back to the end of the dataset if there is no next recommendation timestamp

window_end = model_df["next_recommendation_tstamp"].fillna(dataset_end)  # Fill missing values with the end of the dataset
model_df["follow_up_days"] = (window_end - model_df["recommendation_tstamp"]).dt.total_seconds() / 86400  # in days

In [21]:
model_df.head()

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed,movie_popularity,user_activity,genre_bucket,next_recommendation_tstamp,follow_up_days
0,377084,924,4.303385,2023-03-01 06:10:51,False,7.218910,6.827629,Adventure,2023-03-03 00:53:27,1.779583
1,377084,1201,4.215998,2023-03-01 06:10:51,False,6.855409,6.827629,Action,2023-03-03 00:53:27,1.779583
2,377084,1204,4.236355,2023-03-01 06:10:51,False,6.171701,6.827629,Adventure,2023-05-24 02:12:45,83.834653
3,377084,1258,4.241269,2023-03-01 06:10:51,False,7.410347,6.827629,Horror,2023-03-03 00:53:27,1.779583
4,377084,1732,4.255416,2023-03-01 06:10:51,False,7.380256,6.827629,Comedy,2023-03-03 00:53:27,1.779583


In [22]:
# Feature: recommendation frequency

# Get the rolling count of recommendations for each user-movie pair
rec_sorted["recommendation_frequency"] = (
    rec_sorted.groupby(["user_id", "movie_id"]).cumcount()
)

model_df = model_df.merge(
    rec_sorted[["user_id", "movie_id", "recommendation_tstamp", "recommendation_frequency"]],
    on=["user_id", "movie_id", "recommendation_tstamp"],
    how="left"
)

In [23]:
model_df = model_df.drop(columns=["next_recommendation_tstamp"])

In [24]:
model_df[model_df["recommendation_frequency"] > 0]

,user_id,movie_id,predictedRating,recommendation_tstamp,consumed,movie_popularity,user_activity,genre_bucket,follow_up_days,recommendation_frequency
14,357819,348,5.000000,2023-03-01 07:16:23,False,4.442651,8.430981,Comedy,0.526563,1
15,357819,5373,4.849524,2023-03-01 07:16:23,False,4.077537,8.430981,Drama,0.526563,1
16,357819,41627,5.000000,2023-03-01 07:16:23,False,3.828641,8.430981,Action,0.526563,1
17,357819,176423,4.749099,2023-03-01 07:16:23,False,5.231109,8.430981,Documentary,0.526563,1
18,357819,183423,4.806027,2023-03-01 07:16:23,False,4.852030,8.430981,Documentary,0.526563,1
...,...,...,...,...,...,...,...,...,...,...
1209220,190231,122920,3.797544,2024-04-30 23:59:13,False,7.017506,6.701960,Action,4.907419,13
1209221,190231,122926,3.815679,2024-04-30 23:59:13,False,7.090077,6.701960,Action,4.907419,13
1209222,190231,142488,3.885644,2024-04-30 23:59:13,False,6.727432,6.701960,Thriller,4.907419,13
1209223,190231,187593,3.855397,2024-04-30 23:59:13,False,7.154615,6.701960,Action,4.907419,13


In [25]:
# Check for null values for follow_up_days and recommendation_frequency
null_counts = model_df[["follow_up_days", "recommendation_frequency"]].isnull().sum()
print(f"Null counts for follow_up_days and recommendation_frequency:\n{null_counts}")

Null counts for follow_up_days and recommendation_frequency:
follow_up_days              0
recommendation_frequency    0
dtype: int64


In [26]:
model_df.dtypes

user_id                              int32
movie_id                             int32
predictedRating                    float32
recommendation_tstamp       datetime64[ns]
consumed                              bool
movie_popularity                   float64
user_activity                      float64
genre_bucket                           str
follow_up_days                     float64
recommendation_frequency             int64
dtype: object

In [27]:
model_df["consumed"] = model_df["consumed"].astype(int)  # Convert consumed to int for logistic regression

In [28]:
model_df.dtypes

user_id                              int32
movie_id                             int32
predictedRating                    float32
recommendation_tstamp       datetime64[ns]
consumed                             int64
movie_popularity                   float64
user_activity                      float64
genre_bucket                           str
follow_up_days                     float64
recommendation_frequency             int64
dtype: object

In [29]:
# Set the reference category for genre_bucket to "Drama" - baseline (largest category)
model_df["genre_bucket"] = pd.Categorical(
    model_df["genre_bucket"],
    categories=["Drama"] + [g for g in model_df["genre_bucket"].unique() if g != "Drama"]
)

# Encoding categorical variables: genre_bucket
genre_dummies = pd.get_dummies(model_df["genre_bucket"], drop_first=True)

# Prepare the feature matrix X and target vector y
X = pd.concat([
    model_df[["predictedRating", "movie_popularity", "user_activity",
              "follow_up_days", "recommendation_frequency"]],
    genre_dummies
], axis=1)

X = sm.add_constant(X)  # Add a constant term for the intercept
X = X.astype(float)  # Ensure all features are numeric

y = model_df["consumed"]

In [30]:
logit_model = sm.Logit(y, X).fit()  # Fit the logistic regression model

Optimization terminated successfully.
         Current function value: 0.029827
         Iterations 11


In [31]:
print(logit_model.summary())

                           Logit Regression Results                           
Dep. Variable:               consumed   No. Observations:              1209225
Model:                          Logit   Df Residuals:                  1209208
Method:                           MLE   Df Model:                           16
Date:                Sat, 15 Aug 2026   Pseudo R-squ.:                  0.1041
Time:                        11:28:58   Log-Likelihood:                -36068.
converged:                       True   LL-Null:                       -40258.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -8.5894      0.207    -41.557      0.000      -8.995      -8.184
predictedRating              0.0139      0.031      0.451      0.652      -0.046       0.

In [32]:
np.exp(logit_model.params)  # Calculate the odds ratios for the coefficients

const                       0.000186
predictedRating             1.013979
movie_popularity            1.328653
user_activity               1.233989
follow_up_days              1.007834
recommendation_frequency    0.988359
Adventure                   1.283238
Action                      1.230671
Horror                      1.720251
Comedy                      1.213491
Documentary                 0.648692
Crime                       1.064265
Animation                   1.048714
Unknown                     0.674468
Other                       1.296440
Children                    1.077296
Thriller                    1.143426
dtype: float64

### Odds Ratio (OR) Interpretation

1. **OR = 1** → no change in odds
2. **OR > 1** → higher odds of consumption
3. **OR < 1** → lower odds of consumption

**Examples:**
- **OR = 1.33** → 33% higher odds of consumption
- **OR = 0.53** → 47% lower odds of consumption
- **OR = 1.00** → no change in odds

### Key Finding
- Predicted rating is not statistically significantly associated with consumption (p = 0.65 > 0.05).
    - Since p > 0.05, there is not enough evidence that predicted rating is associated with consumption.
- Movie popularity (OR 1.33) and user activity (OR 1.23)
    - Popular movies and more active users are more likely to consume the movie.
- Follow-up time (OR 1.008/day)
    - Each additional day slightly increases the odds of observing consumption.
- Repeat recommendations (OR 0.988)
    - Recommending the same movie again slightly lowers the chance of consumption.
- Genre (relative to Drama, the reference category)
    - Horror movies have the highest odds of consumption among all genres (OR 1.72, relative to Drama).
    - Documentary (OR 0.65) and Unknown (OR 0.67) are the only genres with lower odds than Drama.